In [1]:
!pip install -q scikit-learn lightgbm


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from lightgbm import LGBMClassifier
from collections import Counter

# ============ 配置 ============
DATA_DIR = "data"
CLEAN_TRAIN = f"{DATA_DIR}/clean_train_v2.csv"
CLEAN_TEST = f"{DATA_DIR}/clean_test_v2.csv"
SAMPLE_SUB = f"{DATA_DIR}/sample_submission.csv"

print("📁 讀取數據...")
clean_train = pd.read_csv(CLEAN_TRAIN, low_memory=False)
clean_test = pd.read_csv(CLEAN_TEST, low_memory=False)
sample_sub = pd.read_csv(SAMPLE_SUB, low_memory=False)

print(f"clean_train: {clean_train.shape}")
print(f"clean_test: {clean_test.shape}")
print(f"sample_sub: {sample_sub.shape}")

📁 讀取數據...
clean_train: (82572, 54)
clean_test: (3084, 51)
sample_sub: (1001, 4)


In [3]:
# ============ 特徵定義 ============
KEY = 'rally_uid'
ACTION_COL = 'next_actionId'
POINT_COL = 'next_pointId'

# 尋找 Task3 目標欄位
RALLY_TARGET_CANDIDATES = ['serverGetPoint', 'server_won_point']
RALLY_TARGET = next((c for c in RALLY_TARGET_CANDIDATES if c in clean_train.columns), None)

# 群組欄位
GROUP_COL = 'match' if 'match' in clean_train.columns else (
    'rally_id' if 'rally_id' in clean_train.columns else None
)

print(f"\n✅ 標籤: {ACTION_COL}, {POINT_COL}, {RALLY_TARGET}")
print(f"✅ 群組: {GROUP_COL}")



✅ 標籤: next_actionId, next_pointId, serverGetPoint
✅ 群組: match


In [4]:
# ============ 特徵選擇 ============
exclude_cols = [
    'match', 'match_id', 'rally_id', 'stroke_number', 'strickNumber',
    'next_actionId', 'next_pointId', 'serverGetPoint', 'server_won_point',
    'rally_uid', 'actionId', 'pointId',
    # 🔴 移除洩漏特徵
    'rally_length', 'rally_progress', 
    'is_early_rally', 'is_mid_rally', 'is_late_rally',
    'score_rally_interact',  # 基於 rally_progress
    'action_point_combo'  # 可能包含當前拍信息
]

# Task1 & Task2
features_task12 = [c for c in clean_train.columns if c not in exclude_cols]

# ============ Task3 特徵（修正版）============

# Task3 只排除「未來信息」，保留「已發生的信息」
exclude_task3 = [
    # 排除的欄位（標籤、ID等）
    'next_actionId', 'next_pointId', 'serverGetPoint', 'server_won_point',
    'rally_uid', 'match', 'match_id', 'rally_id', 'strickNumber',
    
    # 🔴 只排除這些「未來信息」
    'rally_length',           # 總拍數（未來）
    'rally_progress',         # 基於 rally_length
    'is_early_rally',         # 基於 rally_progress
    'is_mid_rally',           # 基於 rally_progress
    'is_late_rally',          # 基於 rally_progress
    'score_rally_interact',   # 基於 rally_progress
]

# Task3 可以使用所有其他特徵
features_task3 = [c for c in clean_train.columns if c not in exclude_task3]


print(f"\n📊 特徵統計:")
print(f"Task1&2 特徵數: {len(features_task12)}")
print(f"Task3 特徵數: {len(features_task3)}")


📊 特徵統計:
Task1&2 特徵數: 37
Task3 特徵數: 41


In [5]:
# ============ 交叉驗證函數 ============
def cv_multiclass_f1(X, y, groups=None, n_splits=5, random_state=42, 
                      class_weight=None, params=None, verbose=True):
    """多分類 Macro F1 交叉驗證"""
    scores = []
    
    if groups is not None:
        gkf = GroupKFold(n_splits=n_splits)
        splits = list(gkf.split(X, y, groups))
    else:
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        splits = list(skf.split(X, y))
    
    if params is None:
        params = {
            'n_estimators': 800,
            'learning_rate': 0.03,
            'num_leaves': 127,
            'max_depth': -1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'min_child_samples': 20,
            'reg_alpha': 0.1,
            'reg_lambda': 0.1,
            'random_state': random_state,
            'n_jobs': -1,
            'verbose': -1
        }
    
    for fold, (train_idx, val_idx) in enumerate(splits, 1):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        model = LGBMClassifier(**params, class_weight=class_weight)
        model.fit(X_tr, y_tr)
        
        y_pred = model.predict(X_va)
        score = f1_score(y_va, y_pred, average='macro')
        scores.append(score)
        
        if verbose:
            print(f"  Fold {fold}: {score:.4f}")
    
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    if verbose:
        print(f"  Mean: {mean_score:.4f} ± {std_score:.4f}")
    
    return mean_score

def cv_binary_auc(X, y, groups=None, n_splits=5, random_state=42, 
                  params=None, verbose=True):
    """二分類 AUC 交叉驗證"""
    scores = []
    
    if groups is not None:
        gkf = GroupKFold(n_splits=n_splits)
        splits = list(gkf.split(X, y, groups))
    else:
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        splits = list(skf.split(X, y))
    
    if params is None:
        params = {
            'n_estimators': 600,
            'learning_rate': 0.05,
            'num_leaves': 63,
            'max_depth': -1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': random_state,
            'n_jobs': -1,
            'verbose': -1
        }
    
    for fold, (train_idx, val_idx) in enumerate(splits, 1):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr)
        
        y_prob = model.predict_proba(X_va)[:, 1]
        score = roc_auc_score(y_va, y_prob)
        scores.append(score)
        
        if verbose:
            print(f"  Fold {fold}: {score:.4f}")
    
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    if verbose:
        print(f"  Mean: {mean_score:.4f} ± {std_score:.4f}")
    
    return mean_score


In [6]:
# ============ Task 1: ActionId（加權版本）============
print("\n" + "="*60)
print("🎯 Task 1: ActionId Prediction (加權訓練)")
print("="*60)

X_action = clean_train[features_task12].copy().fillna(0)
y_action = clean_train[ACTION_COL].astype(int)
groups = clean_train[GROUP_COL] if GROUP_COL else None

# 類別權重
cnt_action = Counter(y_action)
weight_action = {cls: 1.0 / np.sqrt(count) for cls, count in cnt_action.items()}

# ✅ 新增：stroke_number 樣本權重
stroke_weights = np.ones(len(clean_train))
stroke_numbers = clean_train['stroke_number'].values

for i, s in enumerate(stroke_numbers):
    if s <= 3:
        stroke_weights[i] = 2.5  # 早期拍（≤3）權重 2.5 倍
    elif s == 4:
        stroke_weights[i] = 1.5  # 第 4 拍權重 1.5 倍
    else:
        stroke_weights[i] = 0.8  # 後期拍權重降低

print(f"\n📊 樣本權重分佈:")
early = (stroke_numbers <= 3).sum()
mid = (stroke_numbers == 4).sum()
late = (stroke_numbers > 4).sum()
print(f"  早期拍 (≤3): {early} 樣本，權重 2.5x")
print(f"  第 4 拍: {mid} 樣本，權重 1.5x")
print(f"  後期拍 (>4): {late} 樣本，權重 0.8x")
print(f"  加權後有效樣本數: {stroke_weights.sum():.0f}")

# 參數保持原樣
params_action = {
    'n_estimators': 1000,
    'learning_rate': 0.02,
    'num_leaves': 127,
    'max_depth': -1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_samples': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

print("\n開始交叉驗證...")
f1_action_cv = cv_multiclass_f1(
    X_action, y_action, groups, 
    n_splits=5, 
    class_weight=weight_action, 
    params=params_action,
    verbose=True
)

print(f"\n📈 Task1 CV Score: {f1_action_cv:.4f}")

# 訓練最終模型（使用樣本權重）
print("訓練最終模型（加入樣本權重）...")
model_action = LGBMClassifier(**params_action, class_weight=weight_action)
model_action.fit(X_action, y_action, sample_weight=stroke_weights)
print("✅ Task1 模型已使用樣本權重訓練")

# 特徵重要性
importances = pd.Series(
    model_action.feature_importances_, 
    index=X_action.columns
).sort_values(ascending=False)
print("\n🔝 Top 15 重要特徵:")
print(importances.head(15))


🎯 Task 1: ActionId Prediction (加權訓練)



📊 樣本權重分佈:
  早期拍 (≤3): 42898 樣本，權重 2.5x
  第 4 拍: 10268 樣本，權重 1.5x
  後期拍 (>4): 29406 樣本，權重 0.8x
  加權後有效樣本數: 146172

開始交叉驗證...
  Fold 1: 0.1934
  Fold 2: 0.1828
  Fold 3: 0.1977
  Fold 4: 0.1963
  Fold 5: 0.1843
  Mean: 0.1909 ± 0.0062

📈 Task1 CV Score: 0.1909
訓練最終模型（加入樣本權重）...
✅ Task1 模型已使用樣本權重訓練

🔝 Top 15 重要特徵:
action_repeat_count    266739
point_repeat_count     254722
gamePlayerOtherId      203482
gamePlayerId           193466
score_ratio            129844
score_total             99977
score_diff              76459
prev5_actionId          71553
numberGame              70148
prev1_actionId          70120
prev4_actionId          68885
prev2_actionId          67985
prev3_actionId          67732
scoreOther              65521
scoreSelf               57490
dtype: int32


In [7]:
# ============ Task 2: PointId（加權版本）============
print("\n" + "="*60)
print("🎯 Task 2: PointId Prediction (加權訓練)")
print("="*60)

X_point = clean_train[features_task12].copy().fillna(0)
y_point = clean_train[POINT_COL].astype(int)

# 類別權重
cnt_point = Counter(y_point)
weight_point = {cls: 1.0 / np.sqrt(count) for cls, count in cnt_point.items()}

# ✅ stroke_number 樣本權重（與 Task1 相同）
stroke_weights = np.ones(len(clean_train))
stroke_numbers = clean_train['stroke_number'].values

for i, s in enumerate(stroke_numbers):
    if s <= 3:
        stroke_weights[i] = 2.5
    elif s == 4:
        stroke_weights[i] = 1.5
    else:
        stroke_weights[i] = 0.8

print(f"✅ 使用與 Task1 相同的樣本權重策略")

# 參數
params_point = {
    'n_estimators': 800,
    'learning_rate': 0.03,
    'num_leaves': 127,
    'max_depth': -1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_samples': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

print("\n開始交叉驗證...")
f1_point_cv = cv_multiclass_f1(
    X_point, y_point, groups,
    n_splits=5,
    class_weight=weight_point,
    params=params_point,
    verbose=True
)

print(f"\n📈 Task2 CV Score: {f1_point_cv:.4f}")

# 訓練最終模型（使用樣本權重）
print("訓練最終模型（加入樣本權重）...")
model_point = LGBMClassifier(**params_point, class_weight=weight_point)
model_point.fit(X_point, y_point, sample_weight=stroke_weights)
print("✅ Task2 模型已使用樣本權重訓練")


🎯 Task 2: PointId Prediction (加權訓練)
✅ 使用與 Task1 相同的樣本權重策略

開始交叉驗證...
  Fold 1: 0.1654
  Fold 2: 0.1629
  Fold 3: 0.1732
  Fold 4: 0.1712
  Fold 5: 0.1603
  Mean: 0.1666 ± 0.0049

📈 Task2 CV Score: 0.1666
訓練最終模型（加入樣本權重）...
✅ Task2 模型已使用樣本權重訓練


In [8]:
# ============ Task 3: ServerGetPoint（可選加權）============
print("\n" + "="*60)
print("🎯 Task 3: ServerGetPoint Prediction")
print("="*60)

if RALLY_TARGET is not None:
    X_rally = clean_train[features_task3].copy().fillna(0)
    y_rally = clean_train[RALLY_TARGET].astype(int)
    
    # ✅ 可選：也可以加 stroke_number 權重
    stroke_weights = np.ones(len(clean_train))
    stroke_numbers = clean_train['stroke_number'].values
    
    for i, s in enumerate(stroke_numbers):
        if s <= 3:
            stroke_weights[i] = 2.0
        elif s == 4:
            stroke_weights[i] = 1.3
        else:
            stroke_weights[i] = 0.9
    
    # 參數
    params_rally = {
        'n_estimators': 800,
        'learning_rate': 0.03,
        'num_leaves': 63,
        'max_depth': -1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_samples': 20,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    print("開始交叉驗證...")
    auc_rally_cv = cv_binary_auc(
        X_rally, y_rally, groups,
        params=params_rally,
        verbose=True
    )
    
    print(f"\n📈 Task3 CV Score: {auc_rally_cv:.4f}")
    
    # 訓練最終模型（使用樣本權重）
    print("訓練最終模型（加入樣本權重）...")
    model_rally = LGBMClassifier(**params_rally)
    model_rally.fit(X_rally, y_rally, sample_weight=stroke_weights)
    print("✅ Task3 模型已使用樣本權重訓練")
else:
    auc_rally_cv = 0.5
    model_rally = None
    print("⚠️ 找不到 Task3 目標欄位")


🎯 Task 3: ServerGetPoint Prediction
開始交叉驗證...
  Fold 1: 0.6168
  Fold 2: 0.6135
  Fold 3: 0.6317
  Fold 4: 0.6277
  Fold 5: 0.6239
  Mean: 0.6227 ± 0.0067

📈 Task3 CV Score: 0.6227
訓練最終模型（加入樣本權重）...
✅ Task3 模型已使用樣本權重訓練


In [9]:
# ============ 整體評分 ============
print("\n" + "="*60)
print("📊 整體 CV 評分")
print("="*60)
print(f"Task1 (ActionId)  : {f1_action_cv:.4f}")
print(f"Task2 (PointId)   : {f1_point_cv:.4f}")
print(f"Task3 (ServerWin) : {auc_rally_cv:.4f}")

blended_score = 0.4 * f1_action_cv + 0.4 * f1_point_cv + 0.2 * auc_rally_cv
print(f"\n🎯 Blended Score: {blended_score:.4f}")


📊 整體 CV 評分
Task1 (ActionId)  : 0.1909
Task2 (PointId)   : 0.1666
Task3 (ServerWin) : 0.6227

🎯 Blended Score: 0.2675


In [10]:
# ============ 測試集處理與預測 ============
print("\n" + "="*60)
print("🔮 測試集預測")
print("="*60)

# 對齊測試集
test_df = clean_test.copy()
test_df[KEY] = test_df[KEY].astype(str)
sample_sub[KEY] = sample_sub[KEY].astype(str)

# 找出每個 rally_uid 的最後一擊
if 'stroke_number' in test_df.columns:
    dedup = (test_df
             .sort_values([KEY, 'stroke_number'])
             .groupby(KEY, as_index=False)
             .tail(1))
else:
    dedup = test_df.drop_duplicates(subset=[KEY], keep='last')

# 按 sample_sub 順序重排
test_aligned = (dedup.set_index(KEY)
                .reindex(sample_sub[KEY])
                .reset_index())

print(f"對齊後測試集: {len(test_aligned)} 行")

# 特徵準備
features_in_test = [f for f in features_task12 if f in test_aligned.columns]
X_test_action = test_aligned[features_in_test].fillna(0)
X_test_point = X_test_action.copy()

features_task3_in_test = [f for f in features_task3 if f in test_aligned.columns]
X_test_rally = test_aligned[features_task3_in_test].fillna(0)

# 預測
print("開始預測...")
pred_action = model_action.predict(X_test_action)
pred_point = model_point.predict(X_test_point)

if model_rally is not None:
    pred_rally_prob = model_rally.predict_proba(X_test_rally)[:, 1]
else:
    pred_rally_prob = np.full(len(test_aligned), 0.5)





# ============ 生成 Submission ============
print("\n" + "="*60)
print("💾 生成 Submission")
print("="*60)

submission = sample_sub.copy()
submission['actionId'] = pred_action.astype(int)
submission['pointId'] = pred_point.astype(int)
submission['serverGetPoint'] = pred_rally_prob

# 驗證
assert len(submission) == len(sample_sub)
assert (submission[KEY] == sample_sub[KEY]).all()

# 保存
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = f"result/submission_improved_{ts}.csv"
submission.to_csv(out_path, index=False, float_format="%.10f")
print("✅ 已輸出：", out_path)

# 預測分佈檢查
print("\n📊 預測分佈:")
print(f"ActionId 類別數: {submission['actionId'].nunique()}")
print(f"PointId 類別數: {submission['pointId'].nunique()}")
print(f"ServerGetPoint 範圍: [{submission['serverGetPoint'].min():.3f}, {submission['serverGetPoint'].max():.3f}]")
print(f"ServerGetPoint 平均: {submission['serverGetPoint'].mean():.3f}")

print("\n🎉 完成！")


🔮 測試集預測
對齊後測試集: 1001 行
開始預測...

💾 生成 Submission
✅ 已輸出： result/submission_improved_20251102_001238.csv

📊 預測分佈:
ActionId 類別數: 18
PointId 類別數: 11
ServerGetPoint 範圍: [0.032, 0.964]
ServerGetPoint 平均: 0.537

🎉 完成！
